# 02 — Knowledge Graph Construction & Rule-Based Reasoning

Build the RDF knowledge graph, apply logical rules, and inspect derived district features.

In [ ]:
import sys
sys.path.insert(0, '..')

import json
import pandas as pd
import matplotlib.pyplot as plt
import geopandas as gpd

from rdflib import Graph, Namespace
from kg.config import BASE_URI, KG_OUTPUT, SCHEMA_PATH, DISTRICTS_GEOJSON
from kg.build_kg import build_kg, save_kg
from kg.reasoning import compute_all_features, export_features

VN = Namespace(BASE_URI)
%matplotlib inline

## 1. Build the Knowledge Graph

In [ ]:
g = build_kg()
save_kg(g)

print(f'Total triples: {len(g):,}')
print(f'Unique subjects: {len(set(g.subjects())):,}')
print(f'Unique predicates: {len(set(g.predicates())):,}')
print(f'Unique objects: {len(set(g.objects())):,}')

## 2. SPARQL Queries

In [ ]:
# Count entities by type
query = """
PREFIX vn: <http://example.org/vienna/>
PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>

SELECT ?type (COUNT(?s) AS ?count)
WHERE { ?s rdf:type ?type . }
GROUP BY ?type
ORDER BY DESC(?count)
"""
for row in g.query(query):
    print(f'{str(row.type).split("/")[-1]:20s} {int(row.count):>6,}')

In [ ]:
# Which districts are connected to district 1 (Innere Stadt)?
query = """
PREFIX vn: <http://example.org/vienna/>

SELECT ?neighbor ?num
WHERE {
  vn:district/1 vn:connectsTo ?neighbor .
  ?neighbor vn:districtNumber ?num .
}
ORDER BY ?num
"""
print('Districts connected to Innere Stadt (1):')
for row in g.query(query):
    print(f'  District {int(row.num)}')

In [ ]:
# Top districts by stop count
query = """
PREFIX vn: <http://example.org/vienna/>

SELECT ?num ?stops ?routes ?price
WHERE {
  ?d a vn:District ;
     vn:districtNumber ?num ;
     vn:stopCount ?stops ;
     vn:routeCount ?routes ;
     vn:avgPricePerSqm ?price .
}
ORDER BY DESC(?stops)
LIMIT 10
"""
print(f'{"District":>8s} {"Stops":>8s} {"Routes":>8s} {"Avg EUR/m²":>12s}')
for row in g.query(query):
    print(f'{int(row.num):8d} {int(row.stops):8d} {int(row.routes):8d} {float(row.price):12.1f}')

## 3. Rule-Based Features

In [ ]:
features = compute_all_features(g)
export_features(features)

for d in sorted(features):
    f = features[d]
    print(
        f'District {d:2d}: '
        f'access={f["accessibility_score"]:.2f}  '
        f'density={f["transit_density"]:6.1f} ({f["transit_density_label"]:6s})  '
        f'afford={f["affordability_label"]:10s}  '
        f'connect={f["connectivity_score"]:2d}'
    )

## 4. Feature Choropleth Maps

In [ ]:
districts_gdf = gpd.read_file(str(DISTRICTS_GEOJSON))

# Find district number column
for col in ['BEZNR', 'BEZ', 'DISTRICT_CODE', 'district']:
    if col in districts_gdf.columns:
        districts_gdf['d_num'] = districts_gdf[col].astype(int)
        break

feat_df = pd.DataFrame(features).T
feat_df.index = feat_df.index.astype(int)
feat_df.index.name = 'd_num'
merged = districts_gdf.merge(feat_df, on='d_num')

fig, axes = plt.subplots(1, 3, figsize=(18, 6))

merged.plot(column='accessibility_score', ax=axes[0], legend=True, cmap='RdYlGn', edgecolor='white')
axes[0].set_title('Accessibility Score')
axes[0].axis('off')

merged.plot(column='transit_density', ax=axes[1], legend=True, cmap='YlOrRd', edgecolor='white')
axes[1].set_title('Transit Density (stops/km²)')
axes[1].axis('off')

merged.plot(column='avg_price_per_sqm', ax=axes[2], legend=True, cmap='RdYlGn_r', edgecolor='white')
axes[2].set_title('Avg. Price per m²')
axes[2].axis('off')

plt.tight_layout()
plt.show()